# ARL-IDS: Full Training & Evaluation on Kaggle

**Adversarial Reinforcement Learning for IoT Intrusion Detection**

This notebook provides a complete end-to-end workflow:
1. Environment setup
2. Data loading and preprocessing
3. Full training (Encoder + Competitive RL)
4. Comprehensive evaluation
5. Visualization and explainability

---

## 📌 Setup Instructions

1. **GPU Accelerator**: Enable GPU in Kaggle (Settings → Accelerator → GPU)
2. **Dataset**: Upload `train_test_network.csv` to `/kaggle/input/`
3. **Runtime**: Expected ~6-12 hours for 50k episodes

## 1️⃣ Environment Setup

In [ ]:
# Install dependencies
!pip install -q gymnasium torch numpy pandas scikit-learn matplotlib seaborn

import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🔧 Device: {device}")
if device == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2️⃣ Project Setup

Clone the repository or upload source files to Kaggle.

In [ ]:
# Option 1: Clone from GitHub (if public)
# !git clone https://github.com/yourusername/ARL-IDS.git
# %cd ARL-IDS

# Option 2: If files are uploaded to Kaggle, create directory structure
!mkdir -p src/agents src/envs src/representation src/data src/utils src/visualization
!mkdir -p results/checkpoints results/logs results/plots

# Verify structure
!ls -la

## 3️⃣ Upload Source Files

**If not using git clone, manually upload these files:**

Place in `/kaggle/working/src/`:
- `config.py`
- `train.py`
- `evaluate.py`
- `run_Training.py`
- `agents/ddqn_agent.py`
- `agents/attacker_agent.py`
- `envs/adversarial_ids_env.py`
- `representation/encoder.py`
- `data/ton_iot_loader.py`
- `utils/metrics_logger.py`
- `utils/explainability.py`
- `visualization/training_plots.py`

**For now, we'll assume files are already uploaded or use git clone above.**

## 4️⃣ Data Loading & Preprocessing

In [ ]:
# Assuming data is in /kaggle/input/ton-iot-dataset/train_test_network.csv
# Adjust path as needed

import sys
sys.path.insert(0, '/kaggle/working')

from src.data.ton_iot_loader import TonIoTLoader
import numpy as np

# Load data
data_path = "/kaggle/input/ton-iot-dataset/train_test_network.csv"  # Adjust path
print(f"📊 Loading data from: {data_path}")

loader = TonIoTLoader(data_path, seed=42)
X_train, X_test, y_train, y_test = loader.load_and_process()

print(f"\n✅ Data loaded successfully")
print(f"   Train samples: {len(X_train):,}")
print(f"   Test samples: {len(X_test):,}")
print(f"   Features: {X_train.shape[1]}")
print(f"   Classes: {len(np.unique(y_train))}")

# Class distribution
import pandas as pd
class_counts = pd.Series(y_train).value_counts().sort_index()
print(f"\n📈 Class distribution:")
for cls, count in class_counts.items():
    print(f"   Class {cls}: {count:,} ({count/len(y_train)*100:.1f}%)")

## 5️⃣ Full Training Pipeline

### Phase 1: Encoder Pre-training (200 epochs)

In [ ]:
from src.representation.encoder import StateEncoder
from src.config import Config

print("🧠 Phase 1: Training Autoencoder...")
print("   Architecture: 38 → 128 → 64 (latent)")
print("   Training: 200 epochs with balanced sampling\n")

input_dim = X_train.shape[1]
latent_dim = 64
n_classes = len(np.unique(y_train))

encoder = StateEncoder(
    input_dim=input_dim,
    latent_dim=latent_dim,
    n_classes=n_classes,
    learning_rate=1e-3,
    device=device
)

# Train encoder
encoder.train(X_train, y_train, epochs=200, batch_size=64)

# Save encoder
encoder.save("/kaggle/working/results/checkpoints/encoder.pth")
print("\n✅ Encoder training complete and saved!")

### Phase 2: Competitive RL Training (50k episodes)

**⚠️ This will take 6-12 hours on GPU. Monitor the output for balance warnings.**

In [ ]:
import argparse
from src.train import train

print("⚔️  Phase 2: Competitive Training (Defender vs Attacker)")
print("   Episodes: 50,000")
print("   Expected time: 6-12 hours\n")

# Configure training arguments
args = argparse.Namespace(
    data_path=data_path,
    episodes=50000,
    lr=5e-4,
    gamma=0.99,
    epsilon_start=1.0,
    epsilon_end=0.01,
    epsilon_decay=0.9999,
    batch_size=128,
    target_update_freq=100,
    weight_update_freq=500,
    encoder_epochs=200,
    skip_encoder_train=True,  # We already trained it above
    no_reward_shaping=False,
    no_adversary=False,
    no_curriculum=False,
    seed=42
)

# Run training
train(args)

print("\n✅ Competitive training complete!")
print("   Models saved to: /kaggle/working/results/checkpoints/")
print("   Metrics saved to: /kaggle/working/results/logs/")
print("   Plots saved to: /kaggle/working/results/plots/")

## 6️⃣ Training Visualization

View the auto-generated training plots

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt

print("📊 Training Visualizations:\n")

# Display training curves
print("1. Training Dynamics (Defender vs Attacker)")
display(Image("/kaggle/working/results/plots/training_curves.png"))

print("\n2. Per-Class F1 Score Evolution")
display(Image("/kaggle/working/results/plots/class_f1_heatmap.png"))

## 7️⃣ Comprehensive Evaluation

In [ ]:
from src.evaluate import evaluate

print("🧪 Running Joint Evaluation (Defender + Attacker)\n")

eval_args = argparse.Namespace(
    data_path=data_path,
    defender_path="/kaggle/working/results/checkpoints/policy_net.pth",
    attacker_path="/kaggle/working/results/checkpoints/attacker_net.pth",
    encoder_path="/kaggle/working/results/checkpoints/encoder.pth"
)

evaluate(eval_args)

print("\n✅ Evaluation complete!")
print("   Results saved to: /kaggle/working/results/joint_evaluation_results.txt")

## 8️⃣ Display Evaluation Results

In [ ]:
# Read and display results
with open("/kaggle/working/results/joint_evaluation_results.txt", "r") as f:
    results = f.read()

print("="*60)
print("EVALUATION RESULTS")
print("="*60)
print(results)
print("="*60)

## 9️⃣ Explainability Analysis

In [ ]:
from src.utils.explainability import GradientExplainer, visualize_feature_importance
from src.agents.ddqn_agent import DDQNAgent

print("🔍 Generating Feature Importance Analysis\n")

# Load models
state_dim = 64
n_classes = len(np.unique(y_test))

encoder_exp = StateEncoder(input_dim, latent_dim, n_classes, device=device)
encoder_exp.load("/kaggle/working/results/checkpoints/encoder.pth")

defender_exp = DDQNAgent(state_dim, n_classes, device=device)
defender_exp.policy_net.load_state_dict(
    torch.load("/kaggle/working/results/checkpoints/policy_net.pth", map_location=device)
)

# Initialize explainer
explainer = GradientExplainer(encoder_exp, defender_exp, device)

# Explain test samples
print("Analyzing 100 test samples...")
sample_size = min(100, len(X_test))
explanations = explainer.explain_batch(X_test[:sample_size], y_test[:sample_size], top_k=5)

# Calculate accuracy
correct = sum(1 for exp in explanations if exp['correct'])
print(f"\n✅ Batch Accuracy: {correct/sample_size*100:.1f}% ({correct}/{sample_size})")

# Aggregated importance
print("\nComputing aggregated feature importance...")
aggregated = explainer.get_aggregated_importance(X_test[:sample_size], y_test[:sample_size])

print(f"\nTop 10 Most Important Features:")
mean_imp = aggregated['mean_importance']
top_indices = np.argsort(mean_imp)[-10:][::-1]
for i, idx in enumerate(top_indices):
    print(f"  {i+1}. Feature {idx}: {mean_imp[idx]:.3f} ± {aggregated['std_importance'][idx]:.3f}")

# Visualize
visualize_feature_importance(
    mean_imp,
    top_k=15,
    save_path="/kaggle/working/results/plots/feature_importance.png"
)

print("\n✅ Feature importance visualization saved!")

## 🔟 Display Feature Importance

In [ ]:
display(Image("/kaggle/working/results/plots/feature_importance.png"))

## 1️⃣1️⃣ Training Metrics Analysis

In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns

# Load training metrics
metrics_files = !ls /kaggle/working/results/logs/*.json
if metrics_files:
    metrics_file = metrics_files[0]
    print(f"📊 Loading metrics from: {metrics_file}\n")
    
    with open(metrics_file, 'r') as f:
        metrics = json.load(f)
    
    # Summary statistics
    print("Training Summary (Last 100 Episodes):")
    print(f"  Total Episodes: {len(metrics['episodes']):,}")
    print(f"  Avg Defender Score: {np.mean(metrics['defender_scores'][-100:]):.3f}")
    print(f"  Avg Attacker Score: {np.mean(metrics['attacker_scores'][-100:]):.3f}")
    
    # Win rate
    recent_def = metrics['defender_scores'][-100:]
    win_rate = sum(1 for s in recent_def if s > 0) / 100
    print(f"  Defender Win Rate: {win_rate:.1%}")
    
    if metrics['defender_epsilon']:
        print(f"  Final Epsilon (Defender): {metrics['defender_epsilon'][-1]:.4f}")
        print(f"  Final Epsilon (Attacker): {metrics['attacker_epsilon'][-1]:.4f}")
    
    # Warnings
    if metrics['imbalance_warnings']:
        print(f"\n⚠️  Imbalance Warnings: {len(metrics['imbalance_warnings'])}")
        for warning in metrics['imbalance_warnings'][-3:]:
            print(f"     Episode {warning['episode']}: {warning['type']} (win rate: {warning['win_rate']:.1%})")
else:
    print("⚠️  No metrics files found")

## 1️⃣2️⃣ Download Results

Package all results for download

In [ ]:
import shutil

# Create archive
print("📦 Creating results archive...\n")

archive_name = "arl_ids_results"
shutil.make_archive(
    f"/kaggle/working/{archive_name}",
    'zip',
    '/kaggle/working/results'
)

print(f"✅ Results archived to: {archive_name}.zip")
print("\nContents:")
print("  📁 checkpoints/ - Trained models")
print("  📁 logs/ - Training metrics (JSON/CSV)")
print("  📁 plots/ - Visualizations")
print("  📄 joint_evaluation_results.txt - Evaluation report")
print("\nDownload the zip file from the Kaggle output panel →")

## 🎯 Summary

### What We Accomplished:

1. ✅ **Environment Setup** - GPU acceleration enabled
2. ✅ **Data Loading** - ToN_IoT dataset preprocessed
3. ✅ **Encoder Training** - 200 epochs, balanced sampling
4. ✅ **Competitive Training** - 50k episodes (Defender vs Attacker)
5. ✅ **Comprehensive Evaluation** - Clean + adversarial metrics
6. ✅ **Visualizations** - Training curves, F1 heatmaps
7. ✅ **Explainability** - Feature importance analysis
8. ✅ **Results Packaging** - All outputs archived

### Key Metrics:
- See `/kaggle/working/results/joint_evaluation_results.txt` for detailed metrics
- Training visualizations in `/kaggle/working/results/plots/`

### Next Steps:
- Download results archive
- Test on new IoT datasets using transfer learning
- Deploy models for production inference

---

**Thank you for using ARL-IDS! 🚀**